# Live Market Data with ib-engine

Connect to IB, subscribe to SPY, and read live quotes.

**Requirements:** `.env` file with `IB_USERNAME` and `IB_PASSWORD` at project root.

In [1]:
import os
import time
from dotenv import load_dotenv
import ib_engine

load_dotenv()

username = os.environ["IB_USERNAME"]
password = os.environ["IB_PASSWORD"]
print(f"Connecting as {username} (paper)...")

Connecting as your_username (paper)...


In [2]:
# Connect to IB (takes ~21s for CCP auth + farm connections)
t0 = time.time()
engine = ib_engine.connect(username=username, password=password, paper=True)
print(f"Connected in {time.time() - t0:.1f}s")
print(f"Account: {engine.account_id}")

Connected in 24.9s
Account: DUXXXXXXX


In [3]:
# Subscribe to SPY (conId=756733)
spy = engine.subscribe(conid=756733, symbol="SPY")
print(f"SPY instrument ID: {spy}")

# Wait for first ticks to arrive
print("Waiting for market data...")
time.sleep(3)

SPY instrument ID: 0
Waiting for market data...


In [4]:
# Read a quote snapshot (SeqLock read - lock-free)
quote = engine.quote(spy)
print(quote)
print(f"  Bid:    ${quote.bid:.2f}")
print(f"  Ask:    ${quote.ask:.2f}")
print(f"  Last:   ${quote.last:.2f}")
print(f"  Spread: ${quote.ask - quote.bid:.4f}")
print(f"  Volume: {quote.volume:,.0f}")

Quote(bid=4768.83, ask=4087.87, last=681.26)
  Bid:    $4768.83
  Ask:    $4087.87
  Last:   $681.26
  Spread: $-680.9600
  Volume: 0


In [5]:
# Stream quotes for 10 seconds
print("Streaming SPY quotes for 10 seconds...\n")
print(f"{'Time':>8}  {'Bid':>10}  {'Ask':>10}  {'Last':>10}  {'Spread':>8}")
print("-" * 52)

start = time.time()
prev_ts = 0
tick_count = 0

while time.time() - start < 10:
    q = engine.quote(spy)
    if q.timestamp_ns != prev_ts and q.bid > 0:
        elapsed = time.time() - start
        spread = q.ask - q.bid
        print(f"{elapsed:7.2f}s  ${q.bid:9.2f}  ${q.ask:9.2f}  ${q.last:9.2f}  ${spread:7.4f}")
        prev_ts = q.timestamp_ns
        tick_count += 1
    time.sleep(0.01)  # 10ms poll interval

print(f"\n{tick_count} unique quotes in 10 seconds")

Streaming SPY quotes for 10 seconds...

    Time         Bid         Ask        Last    Spread
----------------------------------------------------
   0.00s  $  4768.83  $  4087.87  $   681.26  $-680.9600



1 unique quotes in 10 seconds


In [6]:
# Account state
acct = engine.account()
print(acct)
print(f"  Net Liquidation: ${acct.net_liquidation:,.2f}")
print(f"  Buying Power:    ${acct.buying_power:,.2f}")
print(f"  Unrealized PnL:  ${acct.unrealized_pnl:,.2f}")

AccountState(net_liq=749995.29, buying_power=0)
  Net Liquidation: $749,995.29
  Buying Power:    $0.00
  Unrealized PnL:  $0.00


In [7]:
# Shutdown
engine.shutdown()
print("Engine stopped.")

Engine stopped.
